# Qwen3 ASR + TTS API on Google Colab

这个 notebook 支持两种仓库布局：

- 完整项目仓库：根目录直接包含 `api_server.py`、`asr_engine.py`、`tts_engine.py`
- 当前 `colab/` 子仓库：仓库根目录包含 `colab_runtime.py` / `smoke_test.py`，服务代码放在 `app_bundle/`

它也支持私有 GitHub 仓库克隆，你只需要在配置单元里填 `GITHUB_TOKEN`。

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_REPO_DIR = "/content/drive/MyDrive/qwen3-asr-tts-06b"
REPO_DIR = "/content/qwen3-asr-tts-06b"
REPO_GIT_URL = "https://github.com/john88188/qwen3-asr-tts-06b.git"
REPO_GIT_BRANCH = ""
GITHUB_USERNAME = "x-access-token"
GITHUB_TOKEN = ""

USE_GOOGLE_DRIVE_FOR_MODELS = True
DRIVE_MODELS_ROOT = "/content/drive/MyDrive/qwen-asr-colab-models"
MODELS_ROOT = "/content/models"
HF_TOKEN = ""

RUNTIME_ROOT = "/content/qwen-colab-runtime"
HOST = "0.0.0.0"
PORT = 8000
TORCH_INSTALL_MODE = "auto"  # auto / force / skip
START_CLOUDFLARED_TUNNEL = False

ENABLE_TTS = True
API_KEY = ""
ASR_LOG_LEVEL = "INFO"
TTS_DEFAULT_SPEAKER = "Vivian"
TTS_DEFAULT_LANGUAGE = "Auto"
TTS_MAX_NEW_TOKENS = 128
TTS_MAX_NEW_TOKENS_LIMIT = 512
TTS_ATTN_IMPLEMENTATION = None

In [ ]:
if USE_GOOGLE_DRIVE or USE_GOOGLE_DRIVE_FOR_MODELS:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

repo_dir = Path(REPO_DIR)
drive_repo_dir = Path(DRIVE_REPO_DIR)

def is_supported_workspace(path: Path) -> bool:
    candidates = [
        path / "api_server.py",
        path / "colab_runtime.py",
        path / "colab" / "colab_runtime.py",
        path / "app_bundle" / "api_server.py",
        path / "colab" / "app_bundle" / "api_server.py",
    ]
    return any(candidate.exists() for candidate in candidates)

def build_clone_url(repo_git_url: str, *, github_username: str = "", github_token: str = "") -> str:
    if not github_token:
        return repo_git_url
    parts = urlsplit(repo_git_url)
    if parts.scheme != "https":
        raise RuntimeError("Private GitHub clone requires an https URL")
    username = github_username or "x-access-token"
    userinfo = f"{quote(username, safe='')}:{quote(github_token, safe='')}"
    return urlunsplit((parts.scheme, f"{userinfo}@{parts.netloc}", parts.path, parts.query, parts.fragment))

if is_supported_workspace(repo_dir):
    print(f"Using existing workspace: {repo_dir}")
elif REPO_GIT_URL:
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    clone_url = build_clone_url(
        REPO_GIT_URL,
        github_username=GITHUB_USERNAME,
        github_token=GITHUB_TOKEN,
    )
    cmd = ["git", "clone", "--depth", "1"]
    if REPO_GIT_BRANCH:
        cmd += ["--branch", REPO_GIT_BRANCH]
    cmd += [clone_url, str(repo_dir)]
    display_url = REPO_GIT_URL if not GITHUB_TOKEN else f"{REPO_GIT_URL} [token auth]"
    print("$", "git clone --depth 1", display_url, str(repo_dir))
    subprocess.check_call(cmd)
    print(f"Cloned workspace to: {repo_dir}")
elif DRIVE_REPO_DIR and is_supported_workspace(drive_repo_dir):
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    shutil.copytree(drive_repo_dir, repo_dir)
    print(f"Copied workspace from Drive to: {repo_dir}")
else:
    raise RuntimeError(
        "Workspace source not found. Set REPO_GIT_URL, or put the repo under DRIVE_REPO_DIR, or pre-create REPO_DIR."
    )

workspace_dir = repo_dir.resolve()
os.chdir(workspace_dir)
print("workspace_dir=", workspace_dir)

In [ ]:
import importlib
import sys

sys.path.insert(0, str(workspace_dir))

if (workspace_dir / "colab_runtime.py").exists():
    colab_runtime = importlib.import_module("colab_runtime")
    smoke_test = importlib.import_module("smoke_test")
elif (workspace_dir / "colab" / "colab_runtime.py").exists():
    colab_runtime = importlib.import_module("colab.colab_runtime")
    smoke_test = importlib.import_module("colab.smoke_test")
else:
    raise RuntimeError(f"Unsupported workspace layout: {workspace_dir}")

install_system_packages = colab_runtime.install_system_packages
install_python_packages = colab_runtime.install_python_packages
show_runtime_summary = colab_runtime.show_runtime_summary
resolve_app_dir = colab_runtime.resolve_app_dir
download_models = colab_runtime.download_models
launch_api = colab_runtime.launch_api
read_log_tail = colab_runtime.read_log_tail
stop_service = colab_runtime.stop_service

gpu_summary = smoke_test.gpu_summary
healthz = smoke_test.healthz
tts_speakers = smoke_test.tts_speakers
tts_synthesize_to_file = smoke_test.tts_synthesize_to_file
asr_transcribe_file = smoke_test.asr_transcribe_file

app_dir = resolve_app_dir(workspace_dir)
print("workspace_dir=", workspace_dir)
print("app_dir=", app_dir)

install_system_packages()
install_python_packages(workspace_dir, torch_install_mode=TORCH_INSTALL_MODE)
show_runtime_summary()

In [ ]:
gpu_info = gpu_summary()
gpu_info

In [ ]:
models_root = DRIVE_MODELS_ROOT if USE_GOOGLE_DRIVE_FOR_MODELS else MODELS_ROOT
model_info = download_models(
    models_root=models_root,
    include_tts=ENABLE_TTS,
    hf_token=HF_TOKEN or None,
)
model_info

In [ ]:
service = launch_api(
    repo_dir=app_dir,
    runtime_root=RUNTIME_ROOT,
    models_root=models_root,
    host=HOST,
    port=PORT,
    enable_tts=ENABLE_TTS,
    api_key=API_KEY,
    asr_log_level=ASR_LOG_LEVEL,
    tts_default_speaker=TTS_DEFAULT_SPEAKER,
    tts_default_language=TTS_DEFAULT_LANGUAGE,
    tts_max_new_tokens=TTS_MAX_NEW_TOKENS,
    tts_max_new_tokens_limit=TTS_MAX_NEW_TOKENS_LIMIT,
    tts_attn_implementation=TTS_ATTN_IMPLEMENTATION,
    start_cloudflared_tunnel_flag=START_CLOUDFLARED_TUNNEL,
)
service.to_dict()

In [ ]:
health = healthz(service.local_url, api_key=API_KEY)
health

In [ ]:
from pathlib import Path

tts_output = Path(RUNTIME_ROOT) / "artifacts" / "tts_smoke.wav"
if ENABLE_TTS:
    speakers = tts_speakers(service.local_url, api_key=API_KEY)
    tts_result = tts_synthesize_to_file(
        service.local_url,
        output_path=tts_output,
        text="你好，欢迎使用 Qwen3-TTS。",
        speaker=TTS_DEFAULT_SPEAKER,
        language="zh" if TTS_DEFAULT_LANGUAGE == "Auto" else TTS_DEFAULT_LANGUAGE,
        max_new_tokens=TTS_MAX_NEW_TOKENS,
        api_key=API_KEY,
    )
    display({"speakers": speakers, "tts_result": tts_result})
else:
    print("TTS disabled; skipping TTS smoke test.")

In [ ]:
sample_asr_audio = app_dir / "mnt_data" / "input.wav"
if sample_asr_audio.exists():
    asr_result = asr_transcribe_file(
        service.local_url,
        audio_path=sample_asr_audio,
        language="zh",
        max_new_tokens=256,
        api_key=API_KEY,
    )
    asr_result
else:
    print(f"Sample audio not found, skipping ASR smoke test: {sample_asr_audio}")

In [ ]:
print(read_log_tail(service.log_path, lines=60))
# stop_service(service)